# RAAMove — download & preprocess

*Rhetorical moves in research-article abstracts (8 classes)*

**What it is.** 400 RA abstracts, annotated sentence by sentence with one of eight rhetorical moves (Background, Gap, Purpose, Method, Result, Conclusion, Contribution, Implication). Reported annotator agreement: κ = 0.785.

**Difficulty of the labeling judgment:** ★★☆ — moderate. Moves are functional categories, so neighbouring sentences can be genuinely hard to separate.

**Licence:** CC BY 4.0  
**Cite:** Liu, J. et al. (2024), *LREC-COLING*. github.com/ljk1228/RAAMove

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> This notebook is **generated** from `scripts/reshape.py`. The reshaping code below is the same code `scripts/prep_datasets.py` runs — not a copy of it. If you want to change how the data is reshaped, edit `reshape.py` and re-run `scripts/_generate_download_notebooks.py`.

## Step 1 — Download the raw data

In [ ]:
!git clone --depth 1 https://github.com/ljk1228/RAAMove

## Step 2 — Look at the raw format

This one is **JSON**, split into two files by discipline (`Intelligence.json`, `Engineering.json`). Each record has a `text` and a three-letter move code in `labels`.

In [ ]:
RAW_DIR = "RAAMove"

import json
data = json.loads(open(RAW_DIR + "/Intelligence.json", encoding="utf-8").read())
print("records:", len(data))
data[:3]

## Step 3 — Reshape into the canonical schema

Two decisions:

1. **Expand the codes** — `BAC` → `Background`. A prompt that says "Background" needs far less explaining than one that says "BAC".
2. **Pool the two disciplines** — we treat a move as a rhetorical function rather than a discipline-specific one. That *is* an assumption. Comparing Intelligence against Engineering separately would be a good extension.

In [ ]:
def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

RAAMOVE_LABELS = {'BAC': 'Background', 'GAP': 'Gap', 'MTD': 'Method', 'PUR': 'Purpose', 'RST': 'Result', 'CLN': 'Conclusion', 'CTN': 'Contribution', 'IMP': 'Implication'}

def reshape_raamove(raamove_dir):
    """Read RAAMove's per-domain JSON files and expand the 3-letter move codes.

    The corpus ships two domains (Intelligence, Engineering) as separate files. We pool
    them, because a move is meant to be a rhetorical function rather than a
    discipline-specific one - but that IS an assumption, and comparing the two domains
    separately would be a perfectly good extension.
    """
    source_dir = Path(raamove_dir)
    rows = []
    for filename in ("Intelligence.json", "Engineering.json"):
        path = source_dir / filename
        if not path.exists():
            continue
        data = json.loads(path.read_text(encoding="utf-8"))
        for record in data:
            code = record["labels"]
            if code in RAAMOVE_LABELS:
                label = RAAMOVE_LABELS[code]
            else:
                label = code            # an unexpected code: keep it and let validate() complain
            rows.append({"id": 0, "text": record["text"].strip(), "label": label})
    return reid(rows)

In [ ]:
rows = reshape_raamove(RAW_DIR)

## Step 4 — Check the label balance

Very imbalanced: `Method` is the biggest class by far, and `Implication` has only a couple of dozen sentences. With eight classes and a rare tail, a balanced sample of 7 per class is about the most this pool will support.

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

## A note on what you just built

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is *not* your gold set.

Your gold set comes next, in the project notebook: `sample_pool` draws a *balanced* subset from this pool (equal items per label), which is what makes precision, recall, F1 and the confusion matrix readable. Keeping the two separate also leaves the unsampled items free to serve as few-shot examples without leaking the answers you are testing on.

So: build the pool once, here. Sample from it there.

## Step 5 — Save it

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/raamove_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json

OUT_FILE = "raamove_pool.json"

# In Colab, uncomment these two lines to write straight to your Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/raamove_pool.json"

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)